# 04 · Price optimization (NLopt)

Phase 4 results: profit-maximising **regular** prices for the portfolio.

For each product *i*, with current regular price P₀ and weekly baseline volume B (both from the last 13 weeks), regular-price elasticity β (Phase 2) and unit cost c = (1 − category margin)·P₀:

- **demand:** `Q_i(P) = B_i · (P / P₀_i)^β_i`, the constant-elasticity curve implied by the log-log model;
- **objective:** maximise `Σ (P_i − c_i) · Q_i(P_i)`;
- **constraints**, all written as `fc ≤ 0`:
  - prices within 0.8–1.2 × P₀;
  - portfolio volume ≥ 95% of current;
  - revenue-weighted average price change ≤ +5%;
  - private-label ladders kept, with branded unit prices at least 10% above PL.

The problem is solved with NLopt (SLSQP with analytic gradients, compared with COBYLA and ISRES). Prices are then rounded to "9" endings and repaired so every constraint still holds. Robustness re-solves across 200 posterior elasticity draws.

The linear formula `ΔQ = B·β·ΔP/P₀` (RGMx notebook / article) is also solved and its prices are **scored on the true curve**, to show what the approximation costs. Logic lives in `src/optimization/`; run `make optimize` first.

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import FIGURES, OPTIMIZATION_SUMMARY_PATH, RECOMMENDATIONS_PATH, Config
from src.plots import FigureStore, OptimizationPlotter, PlotStyle
from src.schema import Col, Metric, Scenario, kpi_col

style = PlotStyle()
style.apply()
plotter, store = OptimizationPlotter(style), FigureStore(FIGURES)
bounds = Config.load().optimization.price_bounds
recs = pd.read_parquet(RECOMMENDATIONS_PATH)
summary = pd.read_parquet(OPTIMIZATION_SUMMARY_PATH)

## Solvers and scenarios
Weekly change versus current prices, for the baseline (regular-price) business. `recommended` is the SLSQP optimum rounded to price points; `linear_formula` is the linear-demand optimum scored on the constant-elasticity curve.

In [ ]:
cols = [Col.SCENARIO, "volume_change", "revenue_change", "profit_change", "annual_profit_delta"]
summary[[*cols, Col.EVALUATIONS, Col.FIT_SECONDS, Col.STATUS]].round(4)

In [ ]:
store.save(plotter.scenarios(summary), "04_scenarios")

## Recommended price changes
The dot is the recommended change; the line is the 5th–95th percentile of the optimal price across elasticity draws. A wide line means the elasticity is uncertain, so the price move should be tested before rollout.

In [ ]:
store.save(plotter.price_changes(recs, bounds), "04_price_changes")

## One product, explained
A product's own optimum (▲) can differ from the portfolio recommendation (◆). The +5% average-increase cap makes the optimizer cut prices on elastic, high-revenue products to fund larger increases where profit responds most. The dashed curve is the linear formula: close to the current price, but wrong away from it.

In [ ]:
store.save(plotter.profit_curve(recs, 2066200531, bounds), "04_profit_curve")

## Recommendation table

In [ ]:
price, rec = (
    kpi_col(Scenario.RECOMMENDED, Metric.PRICE),
    kpi_col(Scenario.RECOMMENDED, Metric.PROFIT),
)
view = recs.assign(profit_change=recs[rec] / recs[kpi_col(Scenario.CURRENT, Metric.PROFIT)] - 1)
cols = [
    Col.DESCRIPTION,
    Col.PRODUCT_SIZE,
    Col.ELASTICITY,
    Col.CURRENT_PRICE,
    price,
    Col.PRICE_RATIO,
]
view[[*cols, Col.ROBUST_P05, Col.ROBUST_P95, "profit_change"]].sort_values(Col.PRICE_RATIO).round(3)